In [1]:
import pandas as pd
import rdkit.Chem as Chem
import rdkit.Chem.AllChem as AllChem
from rdkit import DataStructs
from rdchiral.initialization import rdchiralReaction, rdchiralReactants
import numpy as np
import os
import time

import sys
sys.path.append('../core')
from template_extractor import *
from main_v3 import rdchiralRun

/apps/external/apps/jupyter/3.1.1/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/apps/external/apps/jupyter/3.1.1/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [7]:
def load_reference_data(file_path):
    """Load and process reference reaction database."""
    reference_data = pd.read_pickle(file_path)
    
    #if there are multiple atom mapped solutions, then ennumerate
    _id = []
    prod_smiles = []
    rxn_smiles = []
    atom_map_smiles = []
    not_atom_map_smiles = []
    dataset = []

    for row in reference_data.itertuples():
        rxn_list = row[3]
        for list_item in rxn_list:
            _id.append (row [1])
            prod_smiles.append (row[2])
            rxn_smiles.append (list_item)
            atom_map_smiles.append (row[4])
            not_atom_map_smiles.append (row[5]) 

    #save the new dataset as reference_data, to replace the old one
    reference_data = pd.DataFrame({'id': _id,
                            'prod_smiles': prod_smiles,
                            'rxn_smiles': rxn_smiles,
                            'atom mapped smiles-input': atom_map_smiles,
                            'not atom mapped smiles-input':not_atom_map_smiles})
    
    reference_data['prod_fp'] = [calculate_fingerprint(smi) for smi in reference_data['prod_smiles']]
    
    return reference_data

def calculate_fingerprint(smiles):
    """Calculate Morgan fingerprint for a given SMILES string."""

    fp = AllChem.GetMorganFingerprint(Chem.MolFromSmiles(smiles), 2, useChirality=True, useFeatures=True)
    return fp   

def canonicalize_smiles(rxn_smiles):
    """
    Canonicalize reaction SMILES
    """
    try:
        # Split reaction into reactants and products
        reactants, products = rxn_smiles.split('>>')
        # Canonicalize each side
        canon_reactants = '.'.join(sorted([Chem.MolToSmiles(Chem.MolFromSmiles(smi), canonical=True) 
                                         for smi in reactants.split('.')]))
        canon_products = '.'.join(sorted([Chem.MolToSmiles(Chem.MolFromSmiles(smi), canonical=True) 
                                          for smi in products.split('.')]))
        return f"{canon_reactants}>>{canon_products}"
    except:
        return rxn_smiles
    
def _get_precursor_goal(rxn_smiles):
    """Extract and process precursor goal from reaction SMILES."""
    if isinstance(rxn_smiles, list):
        rxn_smiles = rxn_smiles[0]
    reactants = rxn_smiles.split('>')[0]
    prec_goal = Chem.MolFromSmiles(reactants)
    [a.ClearProp('molAtomMapNumber') for a in prec_goal.GetAtoms()]
    prec_goal = Chem.MolToSmiles(prec_goal, True)
    return Chem.MolToSmiles(Chem.MolFromSmiles(prec_goal), True)

def extract_rdenzyme_temp(reference_data):
    """Build cache of RDChiral reactions, templates and fingerprints for each reference reaction."""
    
    reference_data['reaction_smarts'] = None
    reference_data['rcts_ref_fp'] = None 
    issues = []
    
    for j, _ in enumerate(reference_data.index):
        jx = reference_data.index[j]
        current_rhea_id = reference_data['id'][jx]
        rxn_smiles = reference_data['rxn_smiles'][jx]

        # Handle different rxn_smiles formats
        if isinstance(rxn_smiles, list):
            rxn_smiles = rxn_smiles[0]
            
        elif "']" in rxn_smiles:
            rxn_smiles = rxn_smiles[2:-2]

        # Split reaction components
        try:
            rct_0, rea_0, prd_0 = rxn_smiles.split(' ')[0].split('>')
            
            reactants = mols_from_smiles_list(replace_deuterated(rct_0).split('.'))
            products = mols_from_smiles_list(replace_deuterated(prd_0).split('.'))
            
            if reactants is None:
                print(f"Failed to parse reactant SMILES for index {jx}: {rct_0}")
                continue
            if products is None:
                print(f"Failed to parse product SMILES for index {jx}: {prd_0}")
                continue
                
        except ValueError:
            print(f"Warning: Could not split reaction SMILES for index {jx}")
            continue

        # Extract template
        reaction = {
            'reactants': rct_0,
            'products': prd_0,
            '_id': reference_data['id'][jx]
        }
        
        try:
            template = extract_from_reaction(reaction)
            
            if template is None or 'reaction_smarts' not in template:
                print(f"Warning: No valid template generated for index {jx}")
                print(f"Template extraction result: {template}")
                issues.append(jx)
                continue
            
            # Save reactions smarts to dataframe
            reference_data.at[jx, 'reaction_smarts'] = template['reaction_smarts']

            # Get rcts reference fingerprint and save it to dataset
            prec_rxn = _get_precursor_goal(rxn_smiles)
            rcts_ref_fp = calculate_fingerprint(prec_rxn)
            
            reference_data.at[jx, 'rcts_ref_fp'] = rcts_ref_fp
        
        except Exception as e:
            print(f"Error processing reaction {jx}: {str(e)}")
            continue

    print(f"Issues: {len(issues)}")
        
    return reference_data
    

In [10]:
reference_data = load_reference_data("acquired_data/RHEA_atom_mapped_timepoint_7_success.pkl")
saved_temp = extract_rdenzyme_temp(reference_data)

Template extraction result: {'reaction_id': '46622'}
Template extraction result: {'reaction_id': '11402'}
Template extraction result: {'reaction_id': '54034'}
Template extraction result: {'reaction_id': '20789'}
Template extraction result: {'reaction_id': '58326'}
Template extraction result: {'reaction_id': '42798'}
Template extraction result: {'reaction_id': '14339'}
Template extraction result: {'reaction_id': '23314'}
Template extraction result: {'reaction_id': '21757'}
Template extraction result: {'reaction_id': '12297'}
Template extraction result: {'reaction_id': '17623'}
Template extraction result: {'reaction_id': '15014'}
Template extraction result: {'reaction_id': '45198'}
Template extraction result: {'reaction_id': '53474'}
Template extraction result: {'reaction_id': '43226'}
Template extraction result: {'reaction_id': '10509'}
Template extraction result: {'reaction_id': '14042'}
Template extraction result: {'reaction_id': '14042'}
Template extraction result: {'reaction_id': '5

In [11]:
saved_temp.to_pickle('../data/rhea_atom_mapped_for_jx_cache.pkl')